In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
sales = pd.read_csv('/content/drive/MyDrive/online_retail_cleaned_final.csv', parse_dates=["InvoiceDate"])

In [ ]:
sales

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.40
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.00
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.00
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.80
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.00
...,...,...,...,...,...,...,...,...,...
779390,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,12.60
779391,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,16.60
779392,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,16.60
779393,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France,14.85


In [5]:
def build_dataset(sales, cutoff, horizon_days=90):
    cutoff = pd.Timestamp(cutoff)
    end = cutoff + pd.Timedelta(days=horizon_days)

    past   = sales[sales["InvoiceDate"] < cutoff].copy()
    future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < end)]
    past["Amount"] = past["Quantity"] * past["UnitPrice"]

    data = past.groupby("CustomerID").agg(
        last_buy  = ("InvoiceDate", "max"),
        frequency = ("InvoiceNo", "nunique"),
        monetary  = ("Amount", "sum"),
    ).reset_index()

    data["recency"] = (cutoff - data["last_buy"]).dt.days


    data["target"] = data["CustomerID"].isin(future["CustomerID"].unique()).astype(int)
    data["cutoff"] = cutoff
    return data.drop(columns=["last_buy"])

In [7]:
train_cutoffs = ["2010-06-10", "2010-09-10", "2010-12-10", "2011-03-10", "2011-06-10"]
train = pd.concat([build_dataset(sales, c) for c in train_cutoffs], ignore_index=True)
test  = build_dataset(sales, "2011-09-10")
print(train.shape, test.shape)

(20027, 6) (5277, 6)


In [ ]:
train

,CustomerID,frequency,monetary,recency,target,cutoff
0,12346.0,10,230.55,99,1,2010-06-10
1,12349.0,2,1268.52,22,0,2010-06-10
2,12355.0,1,488.21,19,0,2010-06-10
3,12358.0,2,1697.93,2,0,2010-06-10
4,12359.0,4,1522.23,78,1,2010-06-10
...,...,...,...,...,...,...
20022,18283.0,11,1134.89,17,1,2011-06-10
20023,18284.0,1,461.68,248,0,2011-06-10
20024,18285.0,1,427.00,477,0,2011-06-10
20025,18286.0,2,1296.43,293,0,2011-06-10


In [8]:
train['cutoff'].max()

Timestamp('2011-06-10 00:00:00')

In [ ]:
test['cutoff'].max()

Timestamp('2011-09-10 00:00:00')

In [9]:
features = ["recency", "frequency", "monetary"]
valid_cutoff = pd.Timestamp("2011-06-10")

tr = train[train["cutoff"] <  valid_cutoff]
va = train[train["cutoff"] == valid_cutoff]

X_tr, y_tr = tr[features], tr["target"]
X_va, y_va = va[features], va["target"]
print(X_tr.shape, X_va.shape)

(15061, 3) (4966, 3)


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score

dummy = DummyClassifier(strategy="prior").fit(X_tr, y_tr)
print("Dummy AUC:", roc_auc_score(y_va, dummy.predict_proba(X_va)[:, 1]))   # expect 0.5

Dummy AUC: 0.5


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)
print("LogReg AUC:", roc_auc_score(y_va, logreg.predict_proba(X_va)[:, 1]))

LogReg AUC: 0.8124246889383562


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

tree = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_tr, y_tr)
rf   = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=-1).fit(X_tr, y_tr)

for name, m in [("Tree", tree), ("Random Forest", rf)]:
    print(name, "AUC:", roc_auc_score(y_va, m.predict_proba(X_va)[:, 1]))

Tree AUC: 0.8085710220307438
Random Forest AUC: 0.810267289029593


In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                    subsample=0.8, colsample_bytree=0.8, random_state=0)
xgb.fit(X_tr, y_tr)
print("XGB AUC:", roc_auc_score(y_va, xgb.predict_proba(X_va)[:, 1]))

XGB AUC: 0.8105905208850539


In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

def time_cv(model_fn, features, train):
    cutoffs = sorted(train["cutoff"].unique())
    scores = []

    # start from the 3rd cut-off, so there are always at least 2 cut-offs to train on
    for i in range(2, len(cutoffs)):
        v  = cutoffs[i]                        # the cut-off we validate on
        tr = train[train["cutoff"] <  v]       # everything BEFORE it
        va = train[train["cutoff"] == v]       # the validation cut-off

        model = model_fn().fit(tr[features], tr["target"])
        auc = roc_auc_score(va["target"], model.predict_proba(va[features])[:, 1])
        scores.append(auc)

    return scores

In [ ]:
features = ["recency", "frequency", "monetary"]

def make_xgb():
    return XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8, random_state=0)

scores = time_cv(make_xgb, features, train)
print("AUC per fold:", np.round(scores, 3))
print("Mean AUC:", round(np.mean(scores), 3))

AUC per fold: [0.781 0.763 0.811]
Mean AUC: 0.785


In [ ]:
import itertools
import pandas as pd

rows = []
for depth, lr, n in itertools.product([3, 4, 6], [0.03, 0.1], [200, 500]):

    def make_model():
        return XGBClassifier(n_estimators=n, max_depth=depth, learning_rate=lr,
                             subsample=0.8, colsample_bytree=0.8, random_state=0)

    sc = time_cv(make_model, features, train)
    rows.append({"max_depth": depth, "learning_rate": lr, "n_estimators": n,
                 "mean_auc": np.mean(sc), "std": np.std(sc)})

results = pd.DataFrame(rows).sort_values("mean_auc", ascending=False)
print(results.round(4).to_string(index=False))

 max_depth  learning_rate  n_estimators  mean_auc    std
         3           0.03           200    0.7941 0.0159
         4           0.03           200    0.7916 0.0169
         3           0.03           500    0.7890 0.0178
         3           0.10           200    0.7859 0.0190
         6           0.03           200    0.7852 0.0188
         4           0.03           500    0.7848 0.0197
         4           0.10           200    0.7814 0.0202
         3           0.10           500    0.7773 0.0221
         6           0.03           500    0.7750 0.0219
         4           0.10           500    0.7713 0.0230
         6           0.10           200    0.7705 0.0227
         6           0.10           500    0.7531 0.0253


In [ ]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             classification_report, confusion_matrix,
                             precision_score, recall_score, f1_score)

features = ["recency", "frequency", "monetary"]

# 1) train = ALL 5 training cut-offs, test = the Sep 2011 cut-off (never seen before)
X_train, y_train = train[features], train["target"]
X_test,  y_test  = test[features],  test["target"]

# 2) your final model (best settings from time-based validation)
final = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.03,
                      subsample=0.8, colsample_bytree=0.8, random_state=0)
final.fit(X_train, y_train)

proba = final.predict_proba(X_test)[:, 1]     # probability of "will return"

# 3) main scores
print("Test AUC   :", round(roc_auc_score(y_test, proba), 3))
print("Test PR-AUC:", round(average_precision_score(y_test, proba), 3))
print("Base rate (share who really returned):", round(y_test.mean(), 3))




Test AUC   : 0.795
Test PR-AUC: 0.763
Base rate (share who really returned): 0.434


In [ ]:
dummy = DummyClassifier(strategy="prior").fit(X_train, y_train)
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
print("Dummy AUC  :", round(roc_auc_score(y_test, dummy.predict_proba(X_test)[:, 1]), 3))
print("LogReg AUC :", round(roc_auc_score(y_test, logreg.predict_proba(X_test)[:, 1]), 3))

Dummy AUC  : 0.5
LogReg AUC : 0.789


In [ ]:
# 5) precision / recall at the default threshold 0.5
pred = (proba >= 0.5).astype(int)
print(classification_report(y_test, pred, digits=3))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))

              precision    recall  f1-score   support

           0      0.690     0.894     0.779      2988
           1      0.775     0.477     0.590      2289

    accuracy                          0.713      5277
   macro avg      0.733     0.685     0.685      5277
weighted avg      0.727     0.713     0.697      5277

Confusion matrix:
 [[2671  317]
 [1198 1091]]


In [ ]:
rows = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    p = (proba >= t).astype(int)
    rows.append((t, p.mean(), precision_score(y_test, p), recall_score(y_test, p), f1_score(y_test, p)))
print(pd.DataFrame(rows, columns=["threshold", "share_flagged", "precision", "recall", "f1"]).round(3).to_string(index=False))

 threshold  share_flagged  precision  recall    f1
       0.3          0.452      0.671   0.699 0.685
       0.4          0.358      0.719   0.594 0.651
       0.5          0.267      0.775   0.477 0.590
       0.6          0.199      0.830   0.381 0.523
       0.7          0.153      0.868   0.306 0.453


In [ ]:
print(dict(zip(features, final.feature_importances_.round(3))))

{'recency': np.float32(0.333), 'frequency': np.float32(0.476), 'monetary': np.float32(0.191)}


In [ ]:
test_results = test.assign(proba=proba, pred=(proba >= 0.5).astype(int))
wrong = test_results[test_results["pred"] != test_results["target"]]
print(len(wrong))
print(wrong.describe())

1515
         CustomerID    frequency      monetary      recency       target  \
count   1515.000000  1515.000000   1515.000000  1515.000000  1515.000000   
mean   15380.908251     4.189439   1504.144590   163.481848     0.790759   
min    12348.000000     1.000000     16.700000     0.000000     0.000000   
25%    13967.000000     2.000000    423.590000    46.000000     1.000000   
50%    15411.000000     3.000000    850.340000   120.000000     1.000000   
75%    16854.000000     5.000000   1805.495000   281.000000     1.000000   
max    18287.000000    37.000000  26259.110000   646.000000     1.000000   
std     1696.209295     3.738711   2131.444715   138.118414     0.406901   

                    cutoff        proba         pred  
count                 1515  1515.000000  1515.000000  
mean   2011-09-10 00:00:00     0.356583     0.209241  
min    2011-09-10 00:00:00     0.025911     0.000000  
25%    2011-09-10 00:00:00     0.201033     0.000000  
50%    2011-09-10 00:00:00     0.32

In [ ]:
import joblib

all_data = pd.concat([train, test], ignore_index=True)
final.fit(all_data[features], all_data["target"])
joblib.dump(final, "repeat_buyer_model.joblib")

['repeat_buyer_model.joblib']

In [10]:
import xgboost, sklearn

In [11]:
print(xgboost.__version__, sklearn.__version__)

3.4.1 1.6.1
